# Fase 01: Ingesta de Datos (Capa Bronze)
**Objetivo:** Leer los archivos CSV originales del hospital, aplicar esquema tipado, ejecutar auditoría de calidad completa y escribir los datos en formato Parquet en el Data Lake.
**Entradas:** Archivos CSV en `medicalproyect-raw`.
**Salidas:** Archivos Parquet en `medicalproyect-raw/bronze/` + logs de auditoría.

## 1. Inicialización de Spark y Logger

In [ ]:
def _subir_log(sufijo, storage_account):
    try:
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass


# ── Función helper para notificaciones por Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests
        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception:
        pass

# ── Configuración de cuenta de almacenamiento ──
STORAGE_ACCOUNT = "stproyectomastergrupo3"


# ── Inicialización y Logger ──
# Arrancamos el pipeline: importamos las librerías de Spark y logging
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
import logging
import os
from datetime import datetime

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"


# Limpiamos los handlers de logging anteriores para evitar duplicados si re-ejecutamos la celda
# Limpieza de handlers previos (evita duplicados al re‑ejecutar)
for handler in logging.root.handlers[:]:
    handler.flush()
    handler.close()
    logging.root.removeHandler(handler)

# Creamos el logger: escribe en un archivo .log y también muestra en consola
# Configuración del Logger
log_filename = "pipeline_bronze_ingestion.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.Ingestion.Bronze")

logger.info("=" * 60)
logger.info("INICIO DEL PROCESO DE INGESTA — CAPA BRONZE")
logger.info(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 60)

# Iniciamos Spark con AQE activado para optimizar las consultas automáticamente
spark = SparkSession.builder \
    .appName("Medical_Ingestion_Bronze") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

logger.info("SparkSession creada.")
# Subida parcial de log tras esta seccion
_subir_log("bronze/ingesta_bronze_definición_de_esquemas", STORAGE_ACCOUNT)

logger.info("📱 Enviando notificación de inicio...")
_enviar_telegram("🚀 INICIADO: 01 Ingesta Bronze")


## 2. Definición de Esquemas (StructType)
Se definen los tipos de datos esperados para cada tabla. Esto garantiza que la carga sea determinista y que los tipos sean los correctos desde el primer momento.

In [ ]:
# ── Esquemas ──
# Definimos los tipos exactos de cada columna para que Spark no tenga que inferirlos
# Así nos aseguramos de que los datos se lean con el tipo correcto desde el principio
esquema_patients = StructType([
    StructField("patient_id", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("sex", StringType(), True),
    StructField("bmi", DoubleType(), True),
    StructField("systolic_bp", IntegerType(), True),
    StructField("diastolic_bp", IntegerType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("temperature_f", DoubleType(), True),
    StructField("smoking_status", StringType(), True),
    StructField("alcohol_use", StringType(), True),
    StructField("exercise_level", StringType(), True),
    StructField("insurance_type", StringType(), True),
    StructField("charlson_index", IntegerType(), True),
    StructField("dx_hypertension", IntegerType(), True),
    StructField("dx_type2_diabetes", IntegerType(), True),
    StructField("dx_hyperlipidemia", IntegerType(), True),
    StructField("dx_obesity", IntegerType(), True),
    StructField("dx_coronary_artery_disease", IntegerType(), True),
    StructField("dx_heart_failure", IntegerType(), True),
    StructField("dx_atrial_fibrillation", IntegerType(), True),
    StructField("dx_chronic_kidney_disease", IntegerType(), True),
    StructField("dx_copd", IntegerType(), True),
    StructField("dx_asthma", IntegerType(), True),
    StructField("dx_depression", IntegerType(), True),
    StructField("dx_anxiety", IntegerType(), True),
    StructField("dx_hypothyroidism", IntegerType(), True),
    StructField("dx_osteoarthritis", IntegerType(), True),
    StructField("dx_type1_diabetes", IntegerType(), True)
])

esquema_outcomes = StructType([
    StructField("patient_id", StringType(), True),
    StructField("admission_date", DateType(), True),
    StructField("discharge_date", DateType(), True),
    StructField("length_of_stay_days", IntegerType(), True),
    StructField("icu_admission", IntegerType(), True),
    StructField("icu_days", IntegerType(), True),
    StructField("in_hospital_death", IntegerType(), True),
    StructField("discharge_disposition", StringType(), True),
    StructField("readmitted_30d", IntegerType(), True),
    StructField("days_to_readmission", DoubleType(), True),
    StructField("primary_drg", IntegerType(), True),
    StructField("total_charges_usd", DoubleType(), True)
])

esquema_medications = StructType([
    StructField("patient_id", StringType(), True),
    StructField("medication", StringType(), True),
    StructField("dose", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("frequency", StringType(), True),
    StructField("indication", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("duration_days", IntegerType(), True),
    StructField("is_generic", IntegerType(), True),
    StructField("adherence_pct", DoubleType(), True)
])

esquema_lab_results = StructType([
    StructField("patient_id", StringType(), True),
    StructField("test_date", DateType(), True),
    StructField("test_name", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("reference_low", DoubleType(), True),
    StructField("reference_high", DoubleType(), True),
    StructField("flag", StringType(), True),
    StructField("is_abnormal", IntegerType(), True),
    StructField("delta_from_normal", DoubleType(), True)
])

esquema_diagnoses = StructType([
    StructField("patient_id", StringType(), True),
    StructField("visit_date", DateType(), True),
    StructField("visit_type", StringType(), True),
    StructField("primary_diagnosis", StringType(), True),
    StructField("primary_icd10", StringType(), True),
    StructField("secondary_diagnoses", StringType(), True),
    StructField("secondary_icd10s", StringType(), True),
    StructField("provider_specialty", StringType(), True)
])

logger.info("Esquemas definidos para 5 tablas: patients, outcomes, medications, lab_results, diagnoses.")
# Subida parcial de log tras esta seccion
_subir_log("bronze/ingesta_bronze_carga_de_csvs", STORAGE_ACCOUNT)


## 3. Carga de CSVs con Esquema Explícito

In [ ]:
# ── Configuración de Storage y Carga ──
# Configuramos la ruta al Data Lake y el diccionario con los 5 archivos CSV a procesar
STORAGE_ACCOUNT = "stproyectomastergrupo3"
CONTAINER_RAW = "medicalproyect-raw"
CONTAINER_LOGS = "medicalproyect-logs"
base_path = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
ruta_bronze = f"{base_path}/bronze"

logger.info(f"Storage account : {STORAGE_ACCOUNT}")
logger.info(f"Contenedor raw  : {CONTAINER_RAW}")
logger.info(f"Ruta base       : {base_path}")
logger.info("")
logger.info("Iniciando carga de archivos CSV con esquema tipado...")

archivos = {
    "patients":    ("patients.csv",    esquema_patients),
    "outcomes":    ("outcomes.csv",    esquema_outcomes),
    "medications": ("medications.csv", esquema_medications),
    "lab_results": ("lab_results.csv", esquema_lab_results),
    "diagnoses":   ("diagnoses.csv",   esquema_diagnoses),
}

dfs = {}
errores_carga = []

# Recorremos cada archivo CSV, lo leemos con su esquema y contamos los registros
for nombre, (archivo, esquema) in archivos.items():
    try:
        ruta = f"{base_path}/{archivo}"
        dfs[nombre] = spark.read.csv(ruta, header=True, schema=esquema, enforceSchema=True)
        total = dfs[nombre].count()
        logger.info(f"  Cargado '{archivo}' — {total:,} registros.")
    except Exception as e:
        errores_carga.append((archivo, str(e)))
        logger.error(f"  ERROR al cargar '{archivo}': {e}")

if errores_carga:
    logger.error("Se encontraron errores en la carga. Revise los archivos.")
    raise RuntimeError(f"Falló la carga de {len(errores_carga)} archivo(s).")

logger.info(f"Carga completada — {len(dfs)} tablas en memoria.")

# Sacamos cada DataFrame del diccionario a una variable individual para trabajar más cómodo
# Asignar a variables con nombre para uso directo
df_patients    = dfs["patients"]
df_outcomes    = dfs["outcomes"]
df_medications = dfs["medications"]
df_lab_results = dfs["lab_results"]
df_diagnoses   = dfs["diagnoses"]

## 4. Auditoría de Calidad de Datos
Se ejecutan 6 validaciones sobre cada tabla: conteo de filas, tipado vs esquema, nulos/vacíos, duplicados, integridad referencial y rangos clínicos.

In [ ]:
# ── 4.1 Conteo de Filas ──
# Primera auditoría: contamos las filas de cada tabla para verificar que se cargaron todas
import pandas as pd

logger.info("")
logger.info("── Auditoría 1/6: Conteo de filas ──")

conteos = []
for nombre in ["patients", "outcomes", "medications", "lab_results", "diagnoses"]:
    total = dfs[nombre].count()
    conteos.append({"Tabla": nombre, "Filas": total})
    logger.info(f"  {nombre:<15} {total:>10,} filas")

df_conteos = pd.DataFrame(conteos)
display(df_conteos)

In [ ]:
# ── 4.2 Auditoría de Tipado vs Esquema ──
# Verificamos que los tipos de datos reales coincidan con los del esquema que definimos
logger.info("")
logger.info("── Auditoría 2/6: Tipado de columnas vs esquema esperado ──")

esquemas = {
    "patients":    esquema_patients,
    "outcomes":    esquema_outcomes,
    "medications": esquema_medications,
    "lab_results": esquema_lab_results,
    "diagnoses":   esquema_diagnoses,
}

resultados_tipado = []

for nombre, df in dfs.items():
    esquema_esperado = esquemas[nombre]
    tipos_esperados = {f.name: f.dataType.simpleString() for f in esquema_esperado.fields}
    tipos_reales = dict(df.dtypes)

    errores = []
    for col, tipo_esp in tipos_esperados.items():
        if col not in tipos_reales:
            errores.append(f"Falta columna '{col}'")
        elif tipos_reales[col] != tipo_esp:
            errores.append(f"'{col}': esperado={tipo_esp}, real={tipos_reales[col]}")
    for col in tipos_reales:
        if col not in tipos_esperados:
            errores.append(f"Columna extra: '{col}'")

    estado = "OK" if not errores else f"{len(errores)} problema(s)"
    detalle = "; ".join(errores) if errores else "Todos los tipos coinciden"
    resultados_tipado.append({"Tabla": nombre, "Estado": estado, "Detalle": detalle})
    logger.info(f"  {nombre:<15} {estado}")

df_tipado = pd.DataFrame(resultados_tipado)
display(df_tipado)

In [ ]:
# ── 4.3 Auditoría de Nulos, Vacíos y Placeholders ──
# Buscamos nulos, cadenas vacías y placeholders ('null', 'n/a', etc.) que indican datos faltantes
logger.info("")
logger.info("── Auditoría 3/6: Valores nulos, vacíos y placeholders ──")

textos_placeholder = ["undefined", "null", "n/a", "nan", "-", "none", "unknown"]

resultados_nulos = []

for nombre, df in dfs.items():
    total_filas = df.count()
    expresiones_nulos = []

    for nombre_col, tipo_col in df.dtypes:
        condicion = F.isnull(F.col(nombre_col))
        if tipo_col in ('double', 'float'):
            condicion = condicion | F.isnan(F.col(nombre_col))
        col_texto = F.lower(F.trim(F.col(nombre_col).cast("string")))
        condicion = condicion | (col_texto == '') | col_texto.isin(textos_placeholder)
        expresiones_nulos.append(
            F.sum(F.when(condicion, 1).otherwise(0)).alias(nombre_col)
        )

    conteo = df.select(expresiones_nulos).collect()[0].asDict()
    columnas_afectadas = {k: v for k, v in conteo.items() if v and v > 0}
    total_nulos = sum(columnas_afectadas.values())

    if total_nulos > 0:
        detalle = ", ".join([f"{c} ({n})" for c, n in columnas_afectadas.items()])
        logger.info(f"  {nombre:<15} {total_nulos} nulos/vacíos en {len(columnas_afectadas)} columna(s)")
        for c, n in columnas_afectadas.items():
            logger.info(f"    └ {c}: {n}")
    else:
        detalle = "Limpio"
        logger.info(f"  {nombre:<15} Sin nulos ni vacíos.")

    resultados_nulos.append({
        "Tabla": nombre,
        "Filas": total_filas,
        "Nulos/Vacíos": total_nulos,
        "Detalle": detalle
    })

df_nulos = pd.DataFrame(resultados_nulos)
display(df_nulos)

In [ ]:
# ── 4.4 Auditoría de Duplicados ──
# Detectamos filas duplicadas restando las filas únicas del total de cada tabla
logger.info("")
logger.info("── Auditoría 4/6: Duplicados ──")

resultados_duplicados = []

for nombre, df in dfs.items():
    total = df.count()
    unicos = df.dropDuplicates().count()
    duplicados = total - unicos
    pct = (duplicados / total * 100) if total > 0 else 0

    estado = "Sin duplicados" if duplicados == 0 else f"{duplicados} duplicados ({pct:.2f}%)"
    logger.info(f"  {nombre:<15} {estado}")

    resultados_duplicados.append({
        "Tabla": nombre,
        "Total": total,
        "Únicos": unicos,
        "Duplicados": duplicados,
        "% Duplicados": round(pct, 2)
    })

df_duplicados = pd.DataFrame(resultados_duplicados)
display(df_duplicados)

In [ ]:
# ── 4.5 Integridad Referencial ──
# Comprobamos que todos los patient_id de las tablas hijas existan en patients (sin huérfanos)
logger.info("")
logger.info("── Auditoría 5/6: Integridad referencial (patient_id) ──")

tablas_hijas = [
    ("outcomes",    df_outcomes),
    ("medications", df_medications),
    ("lab_results", df_lab_results),
    ("diagnoses",   df_diagnoses),
]

resultados_ref = []

for nombre, df_hija in tablas_hijas:
    huerfanos = df_hija.join(df_patients, on="patient_id", how="left_anti").count()
    total = df_hija.count()
    pct = (huerfanos / total * 100) if total > 0 else 0

    if huerfanos > 0:
        logger.warning(f"  {nombre:<15} {huerfanos} registros huérfanos ({pct:.2f}%)")
    else:
        logger.info(f"  {nombre:<15} Integridad referencial correcta.")

    resultados_ref.append({
        "Tabla hija": nombre,
        "Registros": total,
        "Huérfanos": huerfanos,
        "% Huérfanos": round(pct, 2)
    })

df_ref = pd.DataFrame(resultados_ref)
display(df_ref)

In [ ]:
# ── 4.6 Validación de Rangos Clínicos ──
# Revisamos que los valores numéricos (edad, BMI, tensión, etc.) estén dentro de rangos clínicos lógicos
logger.info("")
logger.info("── Auditoría 6/6: Rangos clínicos en patients ──")

rangos_clinicos = {
    "age":           (0,    120),
    "bmi":           (10,   80),
    "systolic_bp":   (60,   250),
    "diastolic_bp":  (30,   150),
    "heart_rate":    (20,   250),
    "temperature_f": (90,   108),
}

resultados_rangos = []

for campo, (minimo, maximo) in rangos_clinicos.items():
    fuera = df_patients.filter(
        F.col(campo).isNotNull() &
        ((F.col(campo) < minimo) | (F.col(campo) > maximo))
    ).count()
    total = df_patients.filter(F.col(campo).isNotNull()).count()
    pct = (fuera / total * 100) if total > 0 else 0

    if fuera > 0:
        logger.warning(f"  {campo:<18} {fuera} valores fuera de [{minimo}, {maximo}] ({pct:.2f}%)")
    else:
        logger.info(f"  {campo:<18} Todos los valores dentro del rango esperado.")

    resultados_rangos.append({
        "Campo": campo,
        "Rango esperado": f"[{minimo}, {maximo}]",
        "Fuera de rango": fuera,
        "% Anómalo": round(pct, 2)
    })

df_rangos = pd.DataFrame(resultados_rangos)
display(df_rangos)
# Subida parcial de log tras esta seccion
_subir_log("bronze/ingesta_bronze_perfilado_estadistico", STORAGE_ACCOUNT)


## 5. Perfilado Estadístico
Para cada columna numérica se calcula: mín, máx, media, mediana (aprox.) y valores distintos. Para columnas de fecha: mín y máx.

In [ ]:
# ── Perfilado Estadístico ──
# Calculamos estadísticas descriptivas de cada columna: valores únicos, mínimo, máximo, media y mediana
logger.info("")
logger.info("── Perfilado estadístico ──")
logger.info("Calculando estadísticas para todas las tablas...")

resultados_perfil = []

for nombre_tabla, df in dfs.items():
    logger.info(f"  Perfilando: {nombre_tabla}...")
    expresiones = []

    for nombre_col, tipo_col in df.dtypes:
        expresiones.append(F.countDistinct(F.col(nombre_col)).alias(f"{nombre_col}_distinct"))

        if tipo_col in ('int', 'integer', 'double', 'float', 'bigint', 'long'):
            expresiones.append(F.min(F.col(nombre_col)).alias(f"{nombre_col}_min"))
            expresiones.append(F.max(F.col(nombre_col)).alias(f"{nombre_col}_max"))
            expresiones.append(F.round(F.mean(F.col(nombre_col)), 2).alias(f"{nombre_col}_mean"))
            expresiones.append(
                F.expr(f"percentile_approx({nombre_col}, 0.5)").alias(f"{nombre_col}_median")
            )
        elif tipo_col in ('date', 'timestamp'):
            expresiones.append(F.min(F.col(nombre_col)).alias(f"{nombre_col}_min"))
            expresiones.append(F.max(F.col(nombre_col)).alias(f"{nombre_col}_max"))

    if expresiones:
        fila = df.select(expresiones).collect()[0].asDict()
        for nombre_col, tipo_col in df.dtypes:
            resultados_perfil.append({
                "Tabla": nombre_tabla,
                "Columna": nombre_col,
                "Tipo": tipo_col,
                "Únicos": fila.get(f"{nombre_col}_distinct"),
                "Mín": fila.get(f"{nombre_col}_min", "-"),
                "Máx": fila.get(f"{nombre_col}_max", "-"),
                "Media": fila.get(f"{nombre_col}_mean", "-"),
                "Mediana": fila.get(f"{nombre_col}_median", "-"),
            })

logger.info("Perfilado completado.")
df_perfil = pd.DataFrame(resultados_perfil)
display(df_perfil)

## 6. Vistas Temporales SQL
Se crean vistas temporales para poder consultar los datos con Spark SQL.

In [ ]:
# ── Vistas Temporales ──
# Creamos vistas SQL temporales para poder consultar los DataFrames con sentencias SQL
df_patients.createOrReplaceTempView("patients")
df_outcomes.createOrReplaceTempView("outcomes")
df_medications.createOrReplaceTempView("medications")
df_lab_results.createOrReplaceTempView("lab_results")
df_diagnoses.createOrReplaceTempView("diagnoses")

logger.info("Vistas temporales SQL creadas: patients, outcomes, medications, lab_results, diagnoses.")

In [ ]:
# ── Consultas SQL de ejemplo ──
# Ejecutamos consultas SQL rápidas para verificar que los datos y las vistas funcionan bien
logger.info("")
logger.info("── Validaciones SQL rápidas ──")

# Muestra de datos
logger.info("Vista previa — patients (5 filas):")
df_patients.show(5, truncate=False)

# Distribución de medicamentos genéricos vs marca
logger.info("Distribución is_generic en medications:")
spark.sql("""
    SELECT is_generic, COUNT(*) AS total
    FROM medications
    GROUP BY is_generic
    ORDER BY is_generic
""").show()

# Conteo de visitas por tipo
logger.info("Distribución de visit_type en diagnoses:")
spark.sql("""
    SELECT visit_type, COUNT(*) AS total
    FROM diagnoses
    GROUP BY visit_type
    ORDER BY total DESC
""").show()

# Pacientes con hipertensión vs sin ella
logger.info("Pacientes con hipertensión:")
spark.sql("""
    SELECT dx_hypertension, COUNT(*) AS total
    FROM patients
    GROUP BY dx_hypertension
    ORDER BY dx_hypertension
""").show()

## 7. Escritura en Capa Bronze (Parquet)
Los datos se escriben en formato Parquet en `medicalproyect-raw/bronze/<tabla>/`, organizados en subdirectorios por tabla. No se bloquea la escritura por presencia de nulos (eso corresponde a la capa Silver).

In [ ]:
# ── Escritura Parquet ──
# Guardamos los DataFrames en formato Parquet en la capa Bronze del Data Lake
logger.info("")
logger.info("── Escritura en capa Bronze (Parquet) ──")
logger.info(f"Ruta destino: {ruta_bronze}")

for nombre, df in dfs.items():
    try:
        ruta_destino = f"{ruta_bronze}/{nombre}"
        df.write.mode("overwrite").parquet(ruta_destino)
        logger.info(f"  {nombre:<15} → {ruta_destino}")
    except Exception as e:
        logger.error(f"  ERROR escribiendo '{nombre}': {e}")
        raise e

logger.info("Capa BRONZE poblada con éxito.")

## 8. Resumen de Auditoría
Tabla consolidada con el veredicto de cada validación.

In [ ]:
# ── Resumen Consolidado de Auditoría ──
# Juntamos todos los resultados de auditoría en una tabla para tener una visión global de la calidad
resumen = []

for nombre in dfs.keys():
    resumen.append({
        "Tabla": nombre,
        "Filas": df_conteos[df_conteos["Tabla"] == nombre]["Filas"].values[0],
        "Tipado": df_tipado[df_tipado["Tabla"] == nombre]["Estado"].values[0],
        "Nulos": df_nulos[df_nulos["Tabla"] == nombre]["Nulos/Vacíos"].values[0],
        "Duplicados": df_duplicados[df_duplicados["Tabla"] == nombre]["Duplicados"].values[0],
    })

# Agregar integridad referencial (solo tablas hijas)
for _, row in df_ref.iterrows():
    for r in resumen:
        if r["Tabla"] == row["Tabla hija"]:
            r["Huérfanos"] = row["Huérfanos"]

logger.info("")
logger.info("── Resumen Consolidado de Auditoría ──")
display(pd.DataFrame(resumen).fillna("-"))

## 9. Persistencia de Logs y Cierre

In [ ]:
# ── Finalización ──
# Subimos el log al Data Lake y cerramos la sesión de Spark para liberar recursos
logger.info("")
logger.info("Guardando archivo de logs en el Data Lake...")

try:
    from notebookutils import mssparkutils
    log_destino = f"abfss://{CONTAINER_LOGS}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/ingesta_bronze_{datetime.now().strftime('%Y%m%d_%H%M')}.log"
    ruta_log_local = f"file://{os.path.abspath(log_filename)}"
    mssparkutils.fs.cp(ruta_log_local, log_destino)
    logger.info(f"Log persistido en: {log_destino}")
except Exception as e:
    logger.warning(f"No se pudo copiar el log al Data Lake: {e}")

logger.info("")
logger.info("=" * 60)
logger.info("PROCESO DE INGESTA BRONZE FINALIZADO")
logger.info(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 60)

logger.info("EXECUTION STATUS: SUCCESS")
_enviar_telegram("✅ FINALIZADO: 01 Ingesta Bronze")
spark.stop()
logger.info("SparkSession detenida.")